# SARS — VLM Analysis Pipeline
**Templum Divi Augusti · Ankara · 25 BCE**

Run cells top to bottom. If the runtime disconnects, re-run from Cell 3 — `resume=True` skips already-completed images.

## Cell 1 — Mount Drive & set up project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── EDIT THIS if your Drive folder is named differently ──
DRIVE_PROJECT = '/content/drive/MyDrive/SARS_COLAB'

os.chdir(DRIVE_PROJECT)
sys.path.insert(0, DRIVE_PROJECT)

print(f'Working directory: {os.getcwd()}')
print('Contents:', os.listdir('.'))

## Cell 2 — Install dependencies
After this cell finishes, **restart the runtime** (Runtime > Restart runtime), then re-run from Cell 1.

In [ ]:
!pip install -q \
    "transformers>=4.50.0" \
    accelerate \
    torch \
    pillow \
    langchain \
    langchain-huggingface \
    langchain-chroma \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    chromadb \
    sentence-transformers \
    pymupdf

print('Done. Now: Runtime > Restart runtime — then re-run all cells from Cell 1.')

## Cell 3 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU — switch runtime to GPU (Runtime > Change runtime type > T4 GPU)')

## Cell 4 — HuggingFace login
Required for Gemma (gated model). Get your token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # shows a text box — paste your HuggingFace token

## Cell 5 — Load vector store
Uses the pre-built ChromaDB from your Drive upload — no rebuild needed.

In [ ]:
from src.retrieval import _load_vectorstore, query as rag_query

vector_store = _load_vectorstore()
count = vector_store._collection.count()
print(f'Vector store loaded: {count} chunks')

# Quick sanity check
test = rag_query('Corinthian capital acanthus volute proportions', vectorstore=vector_store, k=3)
print(f'Test query returned {len(test)} results:')
for r in test:
    print(f'  [{r["category"]}] {r["source"][:50]} — score: {r["relevance_score"]}')

## Cell 6 — Load Gemma-3 VLM
Downloads ~8 GB on first run. Cached in Colab session after that.

In [ ]:
from src.vlm_analysis import load_gemma3_vlm
import src.vlm_analysis as vlm_analysis

processor, model = load_gemma3_vlm()
vlm_analysis._colab_processor = processor
vlm_analysis._colab_model     = model

print('VLM ready.')

## Cell 7 — Check progress

In [ ]:
import json
from pathlib import Path

output_dir = Path('data/analysis/vlm_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

with open('data/conditioning/registry/conditioning_registry.json') as f:
    registry = json.load(f)

eligible = [e for e in registry if 'vlm_analysis' in e.get('suitable_for', [])]
done     = [f for f in output_dir.glob('*_analysis.json')
            if '"error"' not in f.read_text()]
failed   = [f for f in output_dir.glob('*_analysis.json')
            if '"error"' in f.read_text()]

print(f'Eligible : {len(eligible)}')
print(f'Done     : {len(done)}')
print(f'Failed   : {len(failed)}')
print(f'Remaining: {len(eligible) - len(done)}')

## Cell 8 — Run VLM analysis
`resume=True` skips already-completed images — safe to re-run after a disconnect.

## Cell 7b — Fix absolute paths in registry\nThe registry was built on your local machine so `source_path` points to `/home/ece/...`. This cell rewrites those paths to the Colab location before running analysis."

In [ ]:
import json, os
from pathlib import Path

REGISTRY_PATH = 'data/conditioning/registry/conditioning_registry.json'
LOCAL_PREFIX  = '/home/ece/VSCodeProjects/SARS'   # what's stored in registry
COLAB_PREFIX  = DRIVE_PROJECT                      # what it should be on Colab

with open(REGISTRY_PATH) as f:
    registry = json.load(f)

fixed = 0
for entry in registry:
    for key in ('source_path', 'canny_path', 'depth_path'):
        val = entry.get(key)
        if val and val.startswith(LOCAL_PREFIX):
            entry[key] = val.replace(LOCAL_PREFIX, COLAB_PREFIX)
            fixed += 1

with open(REGISTRY_PATH, 'w') as f:
    json.dump(registry, f, indent=2, ensure_ascii=False)

print(f'Remapped {fixed} paths  ({LOCAL_PREFIX} → {COLAB_PREFIX})')
print('Sample source_path:', registry[0].get('source_path'))

In [ ]:
from src.vlm_analysis import run_colab_batch

results = run_colab_batch(
    conditioning_registry_path='data/conditioning/registry/conditioning_registry.json',
    vector_store=vector_store,
    output_dir='data/analysis/vlm_outputs',
    resume=True,
)

## Cell 8b — Plans dimension extraction
Run a second VLM pass on the architectural floor plans in `data/visual_sources/plans/`.
A focused prompt extracts numeric data (column counts, intercolumniation, dimensions).
These numbers then drive the procedural canny generator in the SDXL step — the front
elevation will have exactly 8 columns because we tell ControlNet "draw 8 verticals here"
instead of borrowing Maison Carrée's 6-column canny.

In [ ]:
from src.analyze_plans import run_plans_analysis

plans_results = run_plans_analysis(
    plans_dir='data/visual_sources/plans',
    output_dir='data/analysis/vlm_outputs',
    resume=True,
)

# Show what dimensions we extracted
from src.procedural_canny import load_plan_dims_aggregated
dims = load_plan_dims_aggregated('data/analysis/vlm_outputs')
print('\nAggregated plan dimensions (used by procedural canny):')
for k, v in dims.items():
    print(f'  {k:25} : {v}')

## Cell 9 — Inspect a single result

In [ ]:
import json
from pathlib import Path

output_dir = Path('data/analysis/vlm_outputs')
files = sorted(output_dir.glob('*_analysis.json'))

if files:
    with open(files[0]) as f:
        sample = json.load(f)
    print(f'File            : {files[0].name}')
    print(f'Folder          : {sample.get("source_folder")}')
    print(f'Mosque interf.  : {sample.get("mosque_interference")}')
    print(f'Fabric quality  : {sample.get("roman_fabric_quality")}')
    print(f'Strat confidence: {sample.get("stratigraphic_confidence")}')
    print(f'Pass            : {sample.get("pass")}')
    elements = sample.get('architectural_elements', [])
    print(f'Elements ({len(elements)}):')
    for e in elements:
        print(f'  [{e.get("period")}] {e.get("element")} — {e.get("confidence")}')
else:
    print('No analysis files yet.')

## Cell 10 — Summary of all completed analyses

In [ ]:
from collections import Counter
from pathlib import Path
import json

output_dir = Path('data/analysis/vlm_outputs')
analyses = []
for f in output_dir.glob('*_analysis.json'):
    try:
        d = json.loads(f.read_text())
        if 'error' not in d:
            analyses.append(d)
    except Exception:
        pass

print(f'Completed analyses: {len(analyses)}')
print()
print('Mosque interference distribution:')
for k, v in Counter(a.get('mosque_interference','?') for a in analyses).most_common():
    print(f'  {k:12} : {v}')
print()
print('Roman fabric quality:')
for k, v in Counter(a.get('roman_fabric_quality','?') for a in analyses).most_common():
    print(f'  {k:12} : {v}')
print()
print('Pass type:')
for k, v in Counter(a.get('pass','?') for a in analyses).most_common():
    print(f'  {k:20} : {v}')